In [ ]:
import scarches as sca
#import torch
import scanpy as sc
import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt
import os
import time
import pickle
from itertools import chain
import itertools
from tqdm.auto import tqdm

import warnings
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
warnings.simplefilter(action='ignore', category=DeprecationWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)

# PLOT FINAL CELLTYPE UMAPS

In [ ]:
ref_genomes=['GRCh38-p14-Gencode_v44','GRCh38-p13-Gencode_v33']

adata_dict={}

for ref_genome in ref_genomes[:1]:
    #fn=f'{proc_dir}/{ref_genome}_filtered_batch_corrected_clustered_annotated.h5ad'
    fn=f'{proc_dir}/{ref_genome}_with_intron_filtered_batch_corrected_clustered_annotated.h5ad'
    adata=sc.read_h5ad(fn)

    adata_dict[ref_genome]=adata

### FIG 2. Plot

In [ ]:
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
proc_dir='../../xenium_data/processed_data/baysor_processed_output'


for ref_genome in ref_genomes[:1]:
    
    adata=adata_dict[ref_genome]
    
    cell_cols=['final_high_level_celltype','final_low_level_celltype']

    #adata.obs=adata.obs.drop(columns=['final_high_level_celltypes','final_low_level_celltypes'])
    
    for n,col in enumerate(cell_cols[1:]):
        ncols=1
        nrows=1#int(np.ceil(len(cell_cols)/ncols))
        #fig=plt.figure(figsize=(ncols*5,nrows*4.5))
        width_adj_param=7
        height_adj_param=7
        heigh_width_ratio=height_adj_param/width_adj_param

        for leg_loc in ["on data",'right margin'][1:]:
        
            fig,ax=plt.subplots(1,1,figsize=(ncols*width_adj_param,nrows*height_adj_param))
    
            #ax=fig.add_subplot(nrows,ncols,n+1) 
    
            #suptitle='-'.join([panel,param_set_string,f'\ncells_scale_{scale_param}_asg_conf_{avg_assignment_conf_thr}'])
            
            suptitle=f"scRNA-seq - {col.split('final_')[-1].replace('_',' ').capitalize()}"
            #fig.suptitle(suptitle,fontweight='bold',fontsize=10,y=0.94,x=0.5)
            #leg_loc='right margin'

            ### CREATE COLOR MAPPING FOR THE CLUSTERS
            #. As there are more than 20 clusters, for the N most populous clusters choose distinc colours, for less populous clusters colors that are similar suffices
            # Select distinct colors for top N categories 
            sorted_categories=adata.obs[col].value_counts()
            N_distinct = 16  # Number of highly distinguishable colors
            distinct_cmap = plt.cm.get_cmap('tab20', N_distinct)  # Or try 'hsv', 'gist_ncar'

            # Generate a gradient colormap for remaining categories
            remaining_categories = len(sorted_categories)# - N_distinct
            #remaining_colors = list(mcolors.CSS4_COLORS.values())[:remaining_categories]  # Use Tableau colors
            #remaining_colors = list(mcolors.XKCD_COLORS.values())[:remaining_categories]  # Use Tableau colors
            remaining_cmap = plt.cm.get_cmap('nipy_spectral', remaining_categories)  # Or try 'hsv', 'gist_ncar' 'nipy_spectral'



            # Create color mapping
            clusters_color_mapping = {}

            # Assign gradient-based colors to the remaining categories
            for i, cat in enumerate(sorted_categories[:N_distinct].index):
                #print(i / remaining_categories)
                clusters_color_mapping[cat] = distinct_cmap(i)

            # Assign distinct colors to the top categories
            for i, cat in enumerate(sorted_categories[N_distinct:].index):
                clusters_color_mapping[cat] = remaining_cmap(i)
    
            #if col in cell_cols:
            #   leg_loc="on data"
            sc.pl.umap(adata, 
                           color=col,
                           show=False,
                           palette=clusters_color_mapping,
                           ax=ax,
                           size=18,
                           #vmin=0.5,
                           #vmax=1,
                           s=40,
                           vmin='p1',
                           vmax='p99',
                           alpha=0.8,
                           legend_fontsize=8,
                           legend_fontweight='semibold',
                           legend_loc=leg_loc)
    
            
            ax.set_title(None)
                        
            #ax.set_xlabel('UMAP1',fontsize=12, fontweight='bold',x=0.15*(height_adj_param/width_adj_param))
            #ax.set_ylabel('UMAP2',fontsize=12, fontweight='bold',y=0.15)
            ax.set_xlabel(None)
            ax.set_ylabel(None)
    
             # Adding small x and y axes in the lower-left corner
            ax.spines['bottom'].set_color('black')
            ax.spines['bottom'].set_linewidth(1)
            ax.spines['left'].set_color('black')
            ax.spines['left'].set_linewidth(1)
            
            # Hide top and right spines
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
    
            # Set custom limits for small axes
            # Set custom bounds for equally long axes
            # Set custom limits for small axes
            # Set custom bounds for equally long axes
            x_axis_length = (ax.get_xlim()[1] - ax.get_xlim()[0]) * 0.2  # Define axis length
            x_axis_offset=x_axis_length*0.1
            y_axis_length = (ax.get_ylim()[1] - ax.get_ylim()[0]) * 0.2 * (1/heigh_width_ratio)  # Define axis length
            y_axis_offset=x_axis_offset#y_axis_length*0.1 
            
            x_start, x_end = ax.get_xlim()[0]+x_axis_offset, ax.get_xlim()[0] + x_axis_length
            y_start, y_end = ax.get_ylim()[0]+y_axis_offset, ax.get_ylim()[0] + y_axis_length
            
            ax.spines['bottom'].set_bounds(x_start, x_end)
            ax.spines['left'].set_bounds(y_start, y_end)
            
            # Add arrows to axis tips
            arrow_style = mpatches.ArrowStyle("-|>", head_length=0.5, head_width=0.2)
            ax.annotate('', xy=(x_end+0.8, ax.get_ylim()[0]), xytext=(x_end - 0.001 * x_axis_length, ax.get_ylim()[0]), 
                        arrowprops=dict(arrowstyle=arrow_style, color='black', lw=1))
            ax.annotate('', xy=(ax.get_xlim()[0], y_end+0.8), xytext=(ax.get_xlim()[0], y_end - 0.001 * y_axis_length), 
                        arrowprops=dict(arrowstyle=arrow_style, color='black', lw=1))
    
    
            ## Add label titles
            ax_label_offset=0.8
            ax.text((x_start+x_end)/2, ax.get_ylim()[0]-ax_label_offset,'UMAP1',weight='bold',fontsize=12,
                    horizontalalignment='center',verticalalignment='center_baseline')
            
            ax.text(ax.get_xlim()[0]-ax_label_offset*(heigh_width_ratio), (y_start+y_end)/2, 'UMAP2',weight='bold',fontsize=12, 
                    horizontalalignment='center',rotation_mode='anchor',rotation=90,
                    verticalalignment='center')


            # If there is a legend, remove it and add a customized one
            if leg_loc=='right margin':
                
                ax.legend_.remove()

                
                legend_handles=[Line2D([0],[0],marker="o",color=c,lw=0,label=l,markerfacecolor=c,markersize=7,)
                                    for l, c in zip(list(adata.obs[col].cat.categories), adata.uns[f"{col}_colors"])]
                '''                
                # Make new Legend
                l1 = ax.legend(handles=legend_handles,                
                               frameon=False,
                               ncols=1,
                               loc='upper left',
                               bbox_to_anchor=(1, 0.75),
                               #title="Cell type",
                              )
                '''             
    
            ## SAvefig    
            os.makedirs(os.path.join(proc_dir,'figure_plots','Fig2','UMAP'),exist_ok=True)
            fn=os.path.join(proc_dir,'figure_plots','Fig2','UMAP',f'{ref_genome}_{col}_legend_{leg_loc}_UMAP.png')
            fig.savefig(fn, dpi=300, bbox_inches='tight')
            
            ##### SAVE LEGEND FOR CELL TYPE AND REGIONS AS SEPARATE FIGURES
            if leg_loc=='right margin':
                # Create a new figure for the legend
                legend_fig, legend_ax = plt.subplots(figsize=(12, 12))  # Adjust size as needed
                legend_ax.axis('off')  # Turn off the axis for a clean legend
                l1 = legend_ax.legend(handles=legend_handles,                
                                       frameon=False,
                                       ncols=1,
                                       markerscale=2,
                                       fontsize=25,
                                       loc='center',
                                       #bbox_to_anchor=(1, 0.75),
                                       #title="Cell type",
                                            )

                os.makedirs(os.path.join(proc_dir,'figure_plots','Fig2','UMAP'),exist_ok=True)
                fn=os.path.join(proc_dir,'figure_plots','Fig2','UMAP',f'{ref_genome}_{col}_legend_{leg_loc}_UMAP_legend.png')
                legend_fig.savefig(fn, dpi=300, bbox_inches='tight')
